# Evaluation

Start to launch environmrnts and pull huggingface/github resources.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
FOLDERNAME = "Visual_Anagrams_2024CVPR/"
assert FOLDERNAME is not None, "[!] Enter the foldername."
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

Mounted at /content/drive


In [ ]:
from huggingface_hub import login, HfApi
token = ""
login(token=token)
# api = HfApi()
# print(api.whoami())

In [ ]:
! pip install -q \
  diffusers \
  transformers \
  safetensors \
  sentencepiece \
  accelerate \
  bitsandbytes \
  einops \
  mediapy \
  accelerate \
  open_clip_torch
!pip install -q git+https://github.com/dangeng/visual_anagrams.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import gc
import mediapy as mp

import torch
from diffusers import DiffusionPipeline

from visual_anagrams.views import get_views
from visual_anagrams.samplers import sample_stage_1, sample_stage_2
from visual_anagrams.utils import add_args, save_illusion, save_metadata

device = 'cuda'

def im_to_np(im):
  im = (im / 2 + 0.5).clamp(0, 1)
  im = im.detach().cpu().permute(1, 2, 0).numpy()
  im = (im * 255).round().astype("uint8")
  return im


# Garbage collection function to free memory
def flush():
    gc.collect()
    torch.cuda.empty_cache()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

T5 model for text handle

In [ ]:
from transformers import T5EncoderModel

text_encoder = T5EncoderModel.from_pretrained(
    "DeepFloyd/IF-I-L-v1.0",
    subfolder="text_encoder",
    device_map="auto",
    variant="fp16",
    torch_dtype=torch.float16,
)

pipe = DiffusionPipeline.from_pretrained(
    "DeepFloyd/IF-I-L-v1.0",
    text_encoder=text_encoder,  # pass the previously instantiated text encoder
    unet=None                   # do not use a UNet here, as it uses too much memory
)
print(device)

pipe = pipe.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

model.safetensors.index.fp16.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

model.fp16-00001-of-00002.safetensors:   0%|          | 0.00/9.96G [00:00<?, ?B/s]

model.fp16-00002-of-00002.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


cuda


# Evaluation
## Single CLIP 分数（A 对齐度、C 误对齐度）


In [ ]:
# CLIP For A&C
# current image path + prompt
import torch
import torchvision.transforms as T
from PIL import Image
import open_clip

def compute_clip_scores_for_each_image(image_paths, prompts, device="cuda", clip_model_name="ViT-B-32"):
    import open_clip
    from PIL import Image
    import torch
    import os

    # Load CLIP model
    model, _, preprocess = open_clip.create_model_and_transforms(
        clip_model_name, pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer(clip_model_name)
    model.to(device).eval()

    # Encode prompts
    tokenized = tokenizer(prompts).to(device)
    with torch.no_grad():
        text_features = model.encode_text(tokenized)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    results = []
    for image_path in image_paths:
        image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            image_feature = model.encode_image(image)
            image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)

        # Get similarity scores between image and each prompt
        sim = (image_feature @ text_features.T).squeeze(0)  # shape: [N]

        # A: worst alignment (min similarity)
        A = sim.min().item()

        # C: Concealment (softmax then sum normalized probability mass on diagonal)
        tau = 0.01
        sim_matrix = sim.unsqueeze(0)  # shape: [1, N]
        softmax_sim = torch.nn.functional.softmax(sim_matrix / tau, dim=1)  # shape: [1, N]
        C = softmax_sim.sum().item() / len(prompts)

        results.append({
            "image": os.path.basename(image_path),
            "A": round(A, 4),
            "C": round(C, 4),
        })

    return results




In [ ]:
# usage

# view 2
illusion_configs = [
    {
        "name": "flip.campfire.man",
        "prompts": ["an oil painting of people around a campfire", "an oil painting of an old man"],
        "views": ["identity", "flip"],
    },
    {
        "name": "jigsaw.houseplants.marilyn",
        "prompts": ["an oil painting of houseplants", "an oil painting of marilyn monroe"],
        "views": ["identity", "jigsaw"],
    },
    {
        "name": "inner.einstein.marilyn",
        "prompts": ["an oil painting of albert einstein", "an oil painting of marilyn monroe"],
        "views": ["identity", "inner_circle"],
    },
    {
        "name": "negate.landscape.houseplants",
        "prompts": ["a lithograph of a landscape", "a lithograph of houseplants"],
        "views": ["identity", "negate"],
    },
    {
        "name": "patch.lemur.kangaroo",
        "prompts": ["a pencil sketch of a lemur", "a pencil sketch of a kangaroo"],
        "views": ["identity", "patch_permute"],
    },
    {
        "name": "pixel.duck.rabbit",
        "prompts": ["a mosaic of a duck", "a mosaic of a rabbit"],
        "views": ["identity", "pixel_permute"],
    },
    {
        "name": "skew.tudor.skull",
        "prompts": ["an oil painting of a tudor portrait", "an oil painting of a skull"],
        "views": ["identity", "skew"],
    },
    {
        "name": "skew.taylor.rose",
        "prompts": ["an oil painting of a Taylor Swift", "an oil painting of a rose"],
        "views": ["identity", "skew"],
    },
]

outputs_folder = "/content/drive/MyDrive/Visual_Anagrams_2024CVPR/outputs/"

output_folder_names = [
    "flip.campfire.man_2025-05-16_13-08-17",
    "inner.einstein.marilyn_2025-05-16_14-52-24",
    "inner.einstein.marilyn_2025-05-16_14-53-14",
    "inner.einstein.marilyn_2025-05-16_14-53-43",
    "jigsaw.houseplants.marilyn_2025-05-16_13-24-58",
    "negate.landscape.houseplants_2025-05-17_03-28-52",
    "patch.lemur.kangaroo_2025-05-17_03-39-32",
    "pixel.duck.rabbit_2025-05-17_03-44-24",
    "pixel.duck.rabbit_2025-05-17_03-47-54",
    "skew.taylor.rose_2025-05-17_05-28-16",
    "skew.taylor.rose_2025-05-17_05-29-25",
    "skew.taylor.rose_2025-05-17_05-31-43",
    "skew.tudor.skull_2025-05-17_05-21-07"
]

# view 3
illusion_configs_ = [
    {
        "name": "threeview.bridge.clock.train",
        "prompts": [
            "a pencil sketch of a bridge",
            "a pencil sketch of a clock tower",
            "a pencil sketch of a train"
        ],
        "views": ["identity", "rotate_cw", "rotate_ccw"],
    },
    {
        "name": "threeview.mask.cat.owl",
        "prompts": [
            "an oil painting of a tribal mask",
            "an oil painting of a cat",
            "an oil painting of an owl"
        ],
        "views": ["identity", "patch_permute", "jigsaw"],
    },
    {
        "name": "threeview.planet.eye.vortex",
        "prompts": [
            "a surreal painting of a planet",
            "a surreal painting of an eye",
            "a surreal painting of a swirling vortex"
        ],
        "views": ["identity", "patch_permute", "jigsaw"],
    },
    {
        "name": "threeview.tree.deer.sunset",
        "prompts": [
            "a water color of a tree",
            "a water color of a deer",
            "a water color of a sunset"
        ],
        "views": ["identity", "flip", "square_hinge"],
    },
    {
        "name": "threeview.waterfall.teddy.rabbit",
        "prompts": ["an oil painting of a waterfall", "an oil painting of a teddy bear", "an oil painting of a rabbit"],
        "views": ["identity", "rotate_cw", "rotate_ccw"],
    }
]

output_folder_names_ = [
    "threeview.bridge.clock.train_2025-05-17_16-30-32",
    "threeview.mask.cat.owl_2025-05-17_16-35-40",
    "threeview.planet.eye.vortex_2025-05-17_16-46-01",
    "threeview.tree.deer.sunset_2025-05-17_16-53-30",
    "threeview.waterfall.teddy.rabbit_2025-05-17_14-43-28"
]

In [ ]:
import re, os

def get_png(folder):
  all_png_paths = []
  for subdir, _, files in os.walk(folder):
      for file in files:
          if file.lower().endswith(".png"):
              full_path = os.path.join(subdir, file)
              all_png_paths.append(full_path)
  return all_png_paths

def get_config(output_folder_names, illusion_configs):
  results = []
  for folder in output_folder_names:
      match = re.match(r"([a-z0-9.]+)_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}", folder)
      t_folder = outputs_folder + folder
      png_paths = get_png(t_folder)
      if match:
          base_name = match.group(1)
          matched_config = next((cfg for cfg in illusion_configs if cfg["name"] == base_name), None)
          results.append({
              "folder": t_folder,
              "config_name": base_name,
              "matched": matched_config is not None,
              "prompts": matched_config["prompts"] if matched_config else None,
              "views": matched_config["views"] if matched_config else None,
              "images_paths": png_paths,
          })
  return results

results_view3 = get_config(output_folder_names=output_folder_names_, illusion_configs=illusion_configs_)
results_view2 = get_config(output_folder_names=output_folder_names, illusion_configs=illusion_configs)

if results_view2:
  print("results_view2:")
  for i in results_view2:
    print(i)
if results_view3:
  print("results_view3:")
  for i in results_view3:
    print(i)

results_view2:
{'folder': '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/outputs/flip.campfire.man_2025-05-16_13-08-17', 'config_name': 'flip.campfire.man', 'matched': True, 'prompts': ['an oil painting of people around a campfire', 'an oil painting of an old man'], 'views': ['identity', 'flip'], 'images_paths': ['/content/drive/MyDrive/Visual_Anagrams_2024CVPR/outputs/flip.campfire.man_2025-05-16_13-08-17/2025-05-16_13-08-17_an_oil_painting_of_people_arou_to_an_oil_painting_of_an_old_man_64.png', '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/outputs/flip.campfire.man_2025-05-16_13-08-17/2025-05-16_13-08-17_an_oil_painting_of_people_arou_to_an_oil_painting_of_an_old_man_256.png', '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/outputs/flip.campfire.man_2025-05-16_13-08-17/2025-05-16_13-08-17_an_oil_painting_of_people_arou_to_an_oil_painting_of_an_old_man_1024.png']}
{'folder': '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/outputs/inner.einstein.marilyn_2025-05-16_14-52-24', 'c

In [ ]:
print("CLIP For View2：")
for entry in results_view2:
    if not entry["matched"]:
        print(f"❌ 无法匹配 config: {entry['config_name']}")
        continue

    image_paths = entry["images_paths"]
    prompts = entry["prompts"]

    # 计算分数
    clip_scores = compute_clip_scores_for_each_image(
        image_paths=image_paths,
        prompts=prompts,
        clip_model_name="ViT-B-32",  # 可换为 ViT-L-14 等
        device="cuda"
    )

    print(f"✅ {entry['config_name']}:")
    for i, score in enumerate(clip_scores):
        print(f"  📸 Image {i+1}: A = {score['A']:.4f}, C = {score['C']:.4f}")


CLIP For View2：
✅ flip.campfire.man:
  📸 Image 1: A = 0.1430, C = 0.5000
  📸 Image 2: A = 0.1582, C = 0.5000
  📸 Image 3: A = 0.1654, C = 0.5000
✅ inner.einstein.marilyn:
  📸 Image 1: A = 0.1762, C = 0.5000
  📸 Image 2: A = 0.1969, C = 0.5000
  📸 Image 3: A = 0.1846, C = 0.5000
✅ inner.einstein.marilyn:
  📸 Image 1: A = 0.2517, C = 0.5000
  📸 Image 2: A = 0.2441, C = 0.5000
  📸 Image 3: A = 0.2333, C = 0.5000
✅ inner.einstein.marilyn:
  📸 Image 1: A = 0.1999, C = 0.5000
  📸 Image 2: A = 0.2196, C = 0.5000
  📸 Image 3: A = 0.2268, C = 0.5000
✅ jigsaw.houseplants.marilyn:
  📸 Image 1: A = 0.1552, C = 0.5000
  📸 Image 2: A = 0.1468, C = 0.5000
  📸 Image 3: A = 0.1636, C = 0.5000
✅ negate.landscape.houseplants:
  📸 Image 1: A = 0.2276, C = 0.5000
  📸 Image 2: A = 0.2432, C = 0.5000
  📸 Image 3: A = 0.2501, C = 0.5000
✅ patch.lemur.kangaroo:
  📸 Image 1: A = 0.2837, C = 0.5000
  📸 Image 2: A = 0.2503, C = 0.5000
  📸 Image 3: A = 0.2632, C = 0.5000
✅ pixel.duck.rabbit:
  📸 Image 1: A = 0.263

In [ ]:
print("CLIP For View3：")
for entry in results_view3:
    if not entry["matched"]:
        print(f"❌ 无法匹配 config: {entry['config_name']}")
        continue

    image_paths = entry["images_paths"]
    prompts = entry["prompts"]

    # 计算分数
    clip_scores = compute_clip_scores_for_each_image(
        image_paths=image_paths,
        prompts=prompts,
        clip_model_name="ViT-B-32",  # 可换为 ViT-L-14 等
        device="cuda"
    )

    print(f"✅ {entry['config_name']}:")
    for i, score in enumerate(clip_scores):
        print(f"  📸 Image {i+1}: A = {score['A']:.4f}, C = {score['C']:.4f}")

CLIP For View3：
✅ threeview.bridge.clock.train:
  📸 Image 1: A = 0.2364, C = 0.3333
  📸 Image 2: A = 0.2679, C = 0.3333
  📸 Image 3: A = 0.2761, C = 0.3333
✅ threeview.mask.cat.owl:
  📸 Image 1: A = 0.2296, C = 0.3333
  📸 Image 2: A = 0.2427, C = 0.3333
  📸 Image 3: A = 0.2284, C = 0.3333
✅ threeview.planet.eye.vortex:
  📸 Image 1: A = 0.2264, C = 0.3333
  📸 Image 2: A = 0.2359, C = 0.3333
  📸 Image 3: A = 0.2420, C = 0.3333
✅ threeview.tree.deer.sunset:
  📸 Image 1: A = 0.2560, C = 0.3333
  📸 Image 2: A = 0.2743, C = 0.3333
  📸 Image 3: A = 0.2732, C = 0.3333
✅ threeview.waterfall.teddy.rabbit:
  📸 Image 1: A = 0.2081, C = 0.3333
  📸 Image 2: A = 0.2557, C = 0.3333
  📸 Image 3: A = 0.2574, C = 0.3333


## N*N CLIP Evaluation
把输入的图片和提示词拼接成N*N的矩阵用于计算整体的分数分布

In [ ]:
import torch
import open_clip
from PIL import Image
import os

def compute_clip_scores_full_matrix(image_paths, prompts, device="cuda", clip_model_name="ViT-B-32"):
    """
    Compute full CLIP similarity matrix S (N_images x N_prompts),
    and alignment/concealment scores across all.
    """
    # Load CLIP model
    model, _, preprocess = open_clip.create_model_and_transforms(
        clip_model_name, pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer(clip_model_name)
    model.to(device).eval()

    # Encode prompts
    tokenized = tokenizer(prompts).to(device)
    with torch.no_grad():
        text_features = model.encode_text(tokenized)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # [P, D]

    # Encode images
    image_tensors = []
    image_names = []
    for img_path in image_paths:
        img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
        image_tensors.append(img)
        image_names.append(os.path.basename(img_path))
    images = torch.cat(image_tensors, dim=0)  # [N, C, H, W]

    with torch.no_grad():
        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)  # [N, D]

    # Compute similarity matrix S
    sim_matrix = image_features @ text_features.T  # [N, P]

    # A score: worst alignment among diagonal (assuming N == P or min(N, P))
    diag_len = min(sim_matrix.size(0), sim_matrix.size(1))
    diag_values = torch.diagonal(sim_matrix, 0)[:diag_len]
    A = diag_values.min().item()

    # C score: concealment (average of row and column softmax traces)
    tau = 0.01
    row_softmax = torch.nn.functional.softmax(sim_matrix / tau, dim=1)  # [N, P]
    col_softmax = torch.nn.functional.softmax(sim_matrix.T / tau, dim=1)  # [P, N]
    C_row = torch.trace(row_softmax) / row_softmax.size(0)
    C_col = torch.trace(col_softmax) / col_softmax.size(0)
    C = ((C_row + C_col) / 2).item()

    return {
        "A": round(A, 4),
        "C": round(C, 4),
        "S": sim_matrix.detach().cpu(),
        "image_names": image_names,
        "prompts": prompts
    }


In [ ]:
print("N*N CLIP For View2：")
for entry in results_view2:
    if not entry["matched"]:
        print(f"❌ 无法匹配 config: {entry['config_name']}")
        continue

    image_paths = entry["images_paths"]
    prompts = entry["prompts"]

    N_clip_scores = compute_clip_scores_full_matrix(image_paths[:2], prompts)
    print(f"📊 {entry['config_name']}: A = {N_clip_scores['A']:.4f}, C = {N_clip_scores['C']:.4f}")


N*N CLIP For View2：
📊 flip.campfire.man: A = 0.1582, C = 0.4553
📊 inner.einstein.marilyn: A = 0.1969, C = 0.5128
📊 inner.einstein.marilyn: A = 0.2441, C = 0.3999
📊 inner.einstein.marilyn: A = 0.2196, C = 0.5273
📊 jigsaw.houseplants.marilyn: A = 0.1468, C = 0.5715
📊 negate.landscape.houseplants: A = 0.2432, C = 0.5722
📊 patch.lemur.kangaroo: A = 0.2503, C = 0.5063
📊 pixel.duck.rabbit: A = 0.2653, C = 0.4237
📊 pixel.duck.rabbit: A = 0.2459, C = 0.4696
📊 skew.taylor.rose: A = 0.2439, C = 0.4827
📊 skew.taylor.rose: A = 0.2281, C = 0.4393
📊 skew.taylor.rose: A = 0.2196, C = 0.3638
📊 skew.tudor.skull: A = 0.2141, C = 0.5298


In [ ]:
print("N*N CLIP For View3：")
for entry in results_view3:
    if not entry["matched"]:
        print(f"❌ 无法匹配 config: {entry['config_name']}")
        continue

    image_paths = entry["images_paths"]
    prompts = entry["prompts"]

    N_clip_scores = compute_clip_scores_full_matrix(image_paths[:2], prompts)
    print(f"📊 {entry['config_name']}: A = {N_clip_scores['A']:.4f}, C = {N_clip_scores['C']:.4f}")

N*N CLIP For View3：
📊 threeview.bridge.clock.train: A = 0.2364, C = 0.3420
📊 threeview.mask.cat.owl: A = 0.2427, C = 0.3834
📊 threeview.planet.eye.vortex: A = 0.2264, C = 0.0515
📊 threeview.tree.deer.sunset: A = 0.2743, C = 0.3500
📊 threeview.waterfall.teddy.rabbit: A = 0.2081, C = 0.0245
